# 03 — MySQL & SQL Analysis

Load the cleaned data into MySQL and run business analysis queries.

In [1]:
%pip install mysql-connector-python

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: C:\Users\hemam\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [26]:
from pathlib import Path
import sqlite3
import pandas as pd

# Find CSV
notebook_dir = Path.cwd()

csv_candidates = [
    notebook_dir / "Data/shopsy_db/shopsy_kitchen_products_cleaned.csv",
    notebook_dir.parent / "Data/shopsy_db/shopsy_kitchen_products_cleaned.csv",
    Path("Data/shopsy_db/shopsy_kitchen_products_cleaned.csv")
]

csv_path = next((path for path in csv_candidates if path.exists()), None)

if csv_path is None:
    raise FileNotFoundError(
        "shopsy_kitchen_products_cleaned.csv not found."
    )

# Load CSV
df = pd.read_csv(csv_path)

# Clean column names
df.columns = df.columns.str.lower().str.strip()

print("CSV loaded successfully")
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())

# Create SQLite database
query_conn = sqlite3.connect(":memory:")

# Load dataframe into SQLite
df.to_sql(
    "products",
    query_conn,
    index=False,
    if_exists="replace"
)

print("Products table created successfully")

CSV loaded successfully
Shape: (30, 9)
Columns: ['product_name', 'price', 'discount', 'rating', 'reviews', 'brand', 'capacity', 'material', 'product_url']
Products table created successfully


In [15]:
query = """
SELECT product_name, brand, price, rating, reviews
FROM products
WHERE reviews IS NOT NULL
ORDER BY reviews DESC
LIMIT 10
"""

display(pd.read_sql(query, query_conn))

,product_name,brand,price,rating,reviews
0,koktailkitchan Pack of 12 Plastic Grocery Cont...,koktailkitchan,322,4.1,311.0
1,BELIZZI Pack of 6 Plastic Fridge Container - 1...,BELIZZI,257,4.1,272.0
2,VR Pack of 3 Plastic Grocery Container - 4500 ...,VR,247,4.1,210.0
3,Qtrix Pack of 8 Plastic Grocery Container - 50...,Qtrix,300,4.4,104.0
4,VR Pack of 4 Plastic Fridge Container - 2500 m...,VR,292,4.1,70.0
5,AneriDEALS Pack of 24 Plastic Grocery Containe...,AneriDEALS,423,4.1,34.0
6,DharmikEn Pack of 4 Plastic Fridge Container -...,DharmikEn,222,4.2,29.0
7,Panel's Pack of 6 Glass Grocery Container - 30...,Panel's,330,4.5,27.0
8,VASOYA Pack of 1 Plastic Fridge Container - 27...,VASOYA,161,4.0,25.0
9,OMORTEX Pack of 6 Plastic Tea Coffee & Sugar C...,OMORTEX,170,4.2,24.0


In [16]:
query = """
SELECT product_name, brand, price, rating, reviews
FROM products
WHERE rating IS NOT NULL
  AND reviews IS NOT NULL
ORDER BY rating DESC, reviews DESC
LIMIT 10
"""

display(pd.read_sql(query, query_conn))

,product_name,brand,price,rating,reviews
0,Panel's Pack of 6 Glass Grocery Container - 30...,Panel's,330,4.5,27.0
1,Qtrix Pack of 8 Plastic Grocery Container - 50...,Qtrix,300,4.4,104.0
2,Flatkitch Pack of 6 Plastic Grocery Container ...,Flatkitch,241,4.4,1.0
3,Veksin Pack of 6 Plastic Grocery Container - 1...,Veksin,473,4.3,23.0
4,MegaValue Pack of 6 Plastic Grocery Container ...,MegaValue,242,4.3,11.0
5,SEQUENCE Pack of 12 Plastic Grocery Container ...,SEQUENCE,320,4.3,8.0
6,INNOVIX Pack of 1 Plastic Cereal Dispenser - 2...,INNOVIX,175,4.3,8.0
7,Kivanshi Pack of 16 Plastic Grocery Container ...,Kivanshi,328,4.3,3.0
8,DharmikEn Pack of 4 Plastic Fridge Container -...,DharmikEn,222,4.2,29.0
9,OMORTEX Pack of 6 Plastic Tea Coffee & Sugar C...,OMORTEX,170,4.2,24.0


In [17]:
query = """
SELECT 
    brand,
    COUNT(*) AS product_count,
    ROUND(AVG(price), 2) AS average_price,
    ROUND(AVG(rating), 2) AS average_rating,
    COALESCE(SUM(reviews), 0) AS total_reviews
FROM products
WHERE brand IS NOT NULL
GROUP BY brand
ORDER BY total_reviews DESC
LIMIT 10
"""

display(pd.read_sql(query, query_conn))

,brand,product_count,average_price,average_rating,total_reviews
0,koktailkitchan,1,322.00,4.10,311.0
1,VR,3,270.67,4.13,283.0
2,BELIZZI,1,257.00,4.10,272.0
3,Qtrix,1,300.00,4.40,104.0
4,Veksin,2,410.00,4.15,38.0
5,AneriDEALS,1,423.00,4.10,34.0
6,DharmikEn,1,222.00,4.20,29.0
7,Panel's,1,330.00,4.50,27.0
8,VASOYA,1,161.00,4.00,25.0
9,OMORTEX,1,170.00,4.20,24.0


In [18]:
query = """
SELECT 
    ROUND(AVG(discount), 2) AS average_discount,
    MAX(discount) AS highest_discount,
    MIN(discount) AS lowest_discount,
    COUNT(*) AS products_with_discount
FROM products
WHERE discount IS NOT NULL
"""

display(pd.read_sql(query, query_conn))

,average_discount,highest_discount,lowest_discount,products_with_discount
0,69.17,89,46,30


In [19]:
query = """
SELECT 
    brand,
    COUNT(*) AS product_count,
    ROUND(MIN(price), 2) AS minimum_price,
    ROUND(AVG(price), 2) AS average_price,
    ROUND(MAX(price), 2) AS maximum_price
FROM products
WHERE brand IS NOT NULL
  AND price IS NOT NULL
GROUP BY brand
ORDER BY average_price DESC
LIMIT 10
"""

display(pd.read_sql(query, query_conn))

,brand,product_count,minimum_price,average_price,maximum_price
0,MOOZICO,1,482.0,482.0,482.0
1,AneriDEALS,1,423.0,423.0,423.0
2,Veksin,2,347.0,410.0,473.0
3,Masox Store,1,358.0,358.0,358.0
4,Nabhya,1,353.0,353.0,353.0
5,KIKANII,1,351.0,351.0,351.0
6,Panel's,1,330.0,330.0,330.0
7,Kivanshi,1,328.0,328.0,328.0
8,koktailkitchan,1,322.0,322.0,322.0
9,SEQUENCE,1,320.0,320.0,320.0


In [20]:
query = """
SELECT 
    material,
    COUNT(*) AS product_count,
    ROUND(AVG(price), 2) AS average_price
FROM products
WHERE material IS NOT NULL
GROUP BY material
ORDER BY product_count DESC, average_price DESC
LIMIT 10
"""

display(pd.read_sql(query, query_conn))

,material,product_count,average_price
0,Plastic,19,289.58
1,"Stainless Steel, Plastic, Steel",4,239.75
2,"Glass, Silicone, Wood",1,482.00
3,"Stainless Steel, Glass, Steel",1,330.00
4,"Plastic, Silicone",1,320.00
5,Steel,1,310.00
6,"Stainless Steel, Steel",1,267.00
7,Ceramic,1,225.00
8,"Plastic, Silicone, Wood",1,158.00


In [22]:
query = """
SELECT 
    discount,
    COUNT(*) AS product_count,
    ROUND(AVG(rating), 2) AS average_rating,
    ROUND(AVG(reviews), 2) AS average_reviews
FROM products
WHERE discount IS NOT NULL
GROUP BY discount
ORDER BY discount
"""

display(pd.read_sql(query, query_conn))

,discount,product_count,average_rating,average_reviews
0,46,1,4.10,311.0
1,51,2,4.20,7.5
2,62,2,4.15,29.0
3,64,2,3.65,NaN
4,65,2,4.25,1.5
5,66,4,4.13,15.0
6,67,1,4.30,3.0
7,69,1,4.40,104.0
8,70,1,4.10,70.0
9,71,2,4.20,14.0


In [23]:
query = """
SELECT 
    COUNT(*) AS total_products,
    SUM(CASE WHEN reviews IS NOT NULL THEN 1 ELSE 0 END) AS products_with_reviews,
    SUM(CASE WHEN reviews IS NULL THEN 1 ELSE 0 END) AS products_without_reviews,
    ROUND(
        100 * AVG(
            CASE WHEN reviews IS NOT NULL THEN 1 ELSE 0 END
        ), 
        2
    ) AS review_coverage_percent
FROM products
"""

display(pd.read_sql(query, query_conn))

,total_products,products_with_reviews,products_without_reviews,review_coverage_percent
0,30,26,4,86.67


In [24]:
query = """
SELECT 
    CASE 
        WHEN price < 200 THEN 'Under 200'
        WHEN price < 400 THEN '200 to 399'
        ELSE '400 and above'
    END AS price_band,
    COUNT(*) AS product_count,
    ROUND(AVG(rating), 2) AS average_rating,
    COALESCE(SUM(reviews), 0) AS total_reviews
FROM products
WHERE price IS NOT NULL
GROUP BY price_band
ORDER BY MIN(price)
"""

display(pd.read_sql(query, query_conn))

,price_band,product_count,average_rating,total_reviews
0,Under 200,5,4.00,70.0
1,200 to 399,22,4.10,1093.0
2,400 and above,3,4.17,61.0


In [25]:
query = """
SELECT 
    product_name,
    brand,
    price,
    discount,
    rating,
    reviews,
    ROUND(
        rating * LOG10(COALESCE(reviews, 0) + 1),
        2
    ) AS engagement_score
FROM products
WHERE rating >= 4
  AND price IS NOT NULL
ORDER BY engagement_score DESC, discount DESC
LIMIT 10
"""

display(pd.read_sql(query, query_conn))

,product_name,brand,price,discount,rating,reviews,engagement_score
0,koktailkitchan Pack of 12 Plastic Grocery Cont...,koktailkitchan,322,46,4.1,311.0,10.23
1,BELIZZI Pack of 6 Plastic Fridge Container - 1...,BELIZZI,257,74,4.1,272.0,9.99
2,VR Pack of 3 Plastic Grocery Container - 4500 ...,VR,247,75,4.1,210.0,9.53
3,Qtrix Pack of 8 Plastic Grocery Container - 50...,Qtrix,300,69,4.4,104.0,8.89
4,VR Pack of 4 Plastic Fridge Container - 2500 m...,VR,292,70,4.1,70.0,7.59
5,Panel's Pack of 6 Glass Grocery Container - 30...,Panel's,330,66,4.5,27.0,6.51
6,AneriDEALS Pack of 24 Plastic Grocery Containe...,AneriDEALS,423,78,4.1,34.0,6.33
7,DharmikEn Pack of 4 Plastic Fridge Container -...,DharmikEn,222,62,4.2,29.0,6.20
8,Veksin Pack of 6 Plastic Grocery Container - 1...,Veksin,473,66,4.3,23.0,5.93
9,OMORTEX Pack of 6 Plastic Tea Coffee & Sugar C...,OMORTEX,170,71,4.2,24.0,5.87
